In [2]:
from mlflow.tracking import MlflowClient

MLFOW_TRACKING_URI = "sqlite:////home/ubuntu/Mlops-course/03-experiment-tracking/mlflow.db"

client = MlflowClient(tracking_uri=MLFOW_TRACKING_URI)

In [3]:
client.search_experiments()

[<Experiment: artifact_location='/home/ubuntu/Mlops-course/02-ml-model/mlruns/2', creation_time=1784428392824, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1784428392824, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/home/ubuntu/Mlops-course/02-ml-model/mlruns/1', creation_time=1783744047989, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783744047989, lifecycle_stage='active', name='nyc_taxi_experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1783737023715, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783737023715, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [4]:
client.create_experiment(name="my-cool-experiment")

MlflowException: Experiment(name=my-cool-experiment) already exists. Error: (sqlite3.IntegrityError) UNIQUE constraint failed: experiments.workspace, experiments.name
[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: ('my-cool-experiment', 'default', None, 'active', 1784695203286, 1784695203286)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [5]:
from mlflow.entities import ViewType

runs_con_modelo = client.search_runs(
    experiment_ids='1',
    filter_string="",
    order_by=['start_time DESC'],
    max_results=10
)

In [6]:
import mlflow
for run in runs_con_modelo:
    artifacts = client.list_artifacts(run.info.run_id)
    paths = [a.path for a in artifacts]
    if 'models_mlflow' in paths:
        print(f"✅ run id: {run.info.run_id}, rmse: {run.data.metrics.get('rmse')}")

client.list_artifacts

<bound method MlflowClient.list_artifacts of <mlflow.tracking.client.MlflowClient object at 0x7d37c7412f90>>

In [7]:
import mlflow

mlflow.set_tracking_uri(MLFOW_TRACKING_URI)

In [ ]:
run_id = 'f16636c84de44fd8bee32cac0bc095c2'
model_uri = f'runs:/{run_id}/models_mlflow' 
registered_version = mlflow.register_model(model_uri=model_uri,name='nyc-taxi-regressor')
registered_version

In [9]:
model_uri

'runs:/f16636c84de44fd8bee32cac0bc095c2/models_mlflow'

In [10]:
client.search_registered_models()
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Archived
version: 2, stage: Production


/tmp/ipykernel_163824/92185721.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [ ]:
model_version = registered_version.version
new_stage = "Production"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=True
)

In [12]:
from datetime import datetime

date = datetime.today().timestamp
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=['secondchallenger'], creation_timestamp=1784264232614, current_stage='Production', deployment_job_state=None, description=('The model version 2 was transitioned to Production on <built-in method '
 'timestamp of datetime.datetime object at 0x7d37bcd3e7f0>'), last_updated_timestamp=1784695224029, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='0c0a5ddfffd849cc9e6312f530ee59e0', run_link='', source='models:/m-140c56b99ee540a18a2976fdb590a898', status='READY', status_message=None, tags={'model': 'GradientBoostingRegressor'}, user_id=None, version=2, workspace='default'>

In [13]:
from sklearn.metrics import mean_squared_error
import pandas as pd
from sklearn.feature_extraction import DictVectorizer

def read_dataframe(filename):
    df = pd.read_parquet(filename)
    
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60 )
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

def preprocess(df,dv):
    categorical = ['PULocationID', 'DOLocationID']
    numerical = ['trip_distance']

    train_dicts = df[categorical + numerical].to_dict(orient='records')


    return dv.transform(train_dicts)

def test_model(stage,X_test, y_test,name="nyc-taxi-regressor"):

    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}") 
    y_pred = model.predict(X_test)
    return {"rmse": mean_squared_error(y_test,y_pred)}

In [14]:
df = read_dataframe("../data/green_tripdata_2021-03.parquet")


In [15]:
client.download_artifacts(run_id=run_id, path='preprocessor',dst_path='.')

'/home/ubuntu/Mlops-course/02-ml-model/preprocessor'

In [16]:
import pickle

with open("preprocessor/preprocessor.b","rb") as f_in:
    dv = pickle.load(f_in)

In [17]:
X_test = preprocess(df, dv)

In [18]:
target = 'duration'
y_test = df[target].values

In [19]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)

CPU times: user 164 ms, sys: 6.8 ms, total: 171 ms
Wall time: 177 ms


MlflowException: No such artifact: ''